# 00 Generate mock data

**Purpose.** Make the simulated data every other notebook, the tests, the backtest and the API read. This notebook is the only generator; `make sim` runs it.

**Data.** Real OpenStreetMap transformer sites and building centres (`data/osm/*.csv`, ODbL). Everything else is simulated and every row carries `is_simulated=True`. The dev sample adds a temporary simulated T-Meralco load profile and a simulated map point for `silip-esp32-test`; real IoT registrations and readings remain separate.

**Outcome.** CSVs in `data/sim/` (same names and columns as before, plus new columns and tables). Each scenario below is a named, seeded knob with its source or `[VERIFY]`.

Run a different size with the `SIM_SCALE` environment variable: `test` (writes `tests/fixtures/sim`), `dev` (default, `data/sim`), `large` and `xlarge` (timing only, `data/sim/large`, `data/sim/xlarge`).

In [1]:
import hashlib
import os
import time

import numpy as np
import pandas as pd
from _data import HOLIDAYS, ROOT, TZ

## Parameters
One scale per use. The seed fixes every random draw, so a rerun writes the same files.

In [2]:
SCALES = {
    # name: transformers, days, output folder, transformers with per-home detail, detail days
    "test": dict(transformers=8, days=21, out="tests/fixtures/sim", detail=1, detail_days=7),
    "dev": dict(transformers=50, days=56, out="data/sim", detail=3, detail_days=14),
    "large": dict(transformers=2000, days=14, out="data/sim/large", detail=1, detail_days=2),
    "xlarge": dict(transformers=4000, days=14, out="data/sim/xlarge", detail=1, detail_days=2),
}
SCALE = os.environ.get("SIM_SCALE", "dev")
SEED = int(os.environ.get("SIM_SEED", "7"))
# Simulator-only salt; deployed systems hash with SILIP_SIN_SALT from Secrets Manager.
SALT = os.environ.get("SILIP_SIN_SALT", "dev-salt-change-me")
START = pd.Timestamp("2026-04-01", tz=TZ)  # hot dry season in Metro Manila
config = SCALES[SCALE]
OUT = ROOT / os.environ.get("SIM_OUT", config["out"])
STEPS_PER_DAY = 96
print(f"scale={SCALE} seed={SEED} -> {OUT.relative_to(ROOT).as_posix()}")

scale=dev seed=7 -> data/sim


## Scenarios
Every knob a Meralco engineer would recognise, with where the number comes from. `[VERIFY]` means nobody has checked it against a real source yet; it is a placeholder chosen to make the pattern visible.

In [3]:
SCENARIOS = {
    "calendar": {
        "holidays": HOLIDAYS,  # in _data.py, shared with the feature code
        "home_daytime_extra_kw": 0.25,  # people at home on weekends and holidays [VERIFY]
        "source": "[VERIFY] Proclamation for 2026 holidays; Holy Week from Easter 2026-04-05",
    },
    "weather": {
        "daily_max_heat_index_c": (39.0, 1.5),  # mean, spread [VERIFY] PAGASA heat index bulletins
        "day_night_swing_c": 7.0,  # [VERIFY]
        "hot_spells_per_30_days": 1.5, "hot_spell_days": (3, 5), "hot_spell_extra_c": 4.0,  # [VERIFY]
        "rain_day_share": 0.06, "rain_cooling_c": 5.0,  # dry season, few rainy days [VERIFY]
        "typhoon_days_after_day_40": 1, "typhoon_cooling_c": 8.0,  # late-May typhoon [VERIFY]
        "source": "[VERIFY] PAGASA climate normals for Metro Manila, April to May",
    },
    "aircon": {
        "kw_per_home": (0.8, 1.6), "heat_scale": "clip((heat_index - 30) / 10, 0, 1.5)",
        "source": "silip-domain skill (evening aircon scaled by heat index)",
    },
    "ev": {
        "site_share": 0.2,  # transformers in an EV-adopting neighbourhood [VERIFY]
        "home_share_by_end": {"ev_site": 0.15, "other": 0.01},  # [VERIFY]
        "charger_kw": (3.3, 7.4), "charge_from_hour": 20, "charge_hours": (2, 4),
        "source": "charger kW and 8 PM start: silip-domain skill. Context: EVs were 24.7% of Jan to Aug 2026 vehicle sales (SPEC#sources); home share is [VERIFY]",
    },
    "solar": {
        "home_share": 0.05, "kwp": (2.0, 4.0),  # [VERIFY]
        "source": "[VERIFY] net-metering uptake in the Meralco franchise",
    },
    "jumpers": {
        "transformer_share": 0.1, "kw": (2.0, 8.0),  # unmetered load, evening-heavy [VERIFY]
        "check_meter_share": 0.3,  # which transformers have check meters is an open question in SPEC
        "source": "[VERIFY] SPEC open question: which transformers have check meters",
    },
    "clamps": {
        "coverage": (0.3, 0.9),  # opt-in share of homes per transformer [VERIFY]
        "installed_during_run_share": 0.2, "offline_share": 0.05,  # offline share kept from the old simulator
        "wifi_drops_per_day": 0.03, "wifi_drop_hours": (1, 12), "missing_interval_share": 0.01,
        "late_share": 0.02, "duplicate_share": 0.005, "noise_share": 0.01,  # SCT-013-000 is ±1% (SPEC#hardware)
        "source": "noise: SPEC#hardware (SCT-013-000 ±1%); the rest [VERIFY] from the pilot",
    },
    "transformers": {
        "ratings_kva": [25, 37.5, 50, 75, 100], "homes": (20, 60), "base_kw": (0.2, 0.6),
        # peak loading class: (share, low, high) of the 99th-percentile load / usable rating [VERIFY]
        "loading_mix": {"light": (0.30, 0.50, 0.75), "normal": (0.45, 0.75, 0.95),
                        "tight": (0.15, 0.95, 1.08), "over": (0.10, 1.08, 1.30)},
        "lead_up_share": 0.1, "lead_up_growth": (0.10, 0.20), "lead_up_days": (7, 12),  # [VERIFY]
        "source": "ratings, homes and base kW: silip-domain skill; loading mix and lead-up [VERIFY]",
    },
    "outages": {
        "trip_rule": "load > 110% for 3+ h on 3+ days",  # silip-domain skill
        "other_per_day": 0.05, "rain_multiplier": 3, "typhoon_multiplier": 15,  # [VERIFY]
        "median_minutes": {"overload": 180, "weather": 150, "equipment": 240}, "sigma": 0.5,
        "source": "trip rule and medians: silip-domain skill (restoration lognormal, median 2 to 4 h); weather multipliers [VERIFY]",
    },
}
pd.DataFrame(
    [{"scenario": k, "source": v["source"]} for k, v in SCENARIOS.items()]
).style.hide(axis="index")

scenario,source
calendar,[VERIFY] Proclamation for 2026 holidays; Holy Week from Easter 2026-04-05
weather,"[VERIFY] PAGASA climate normals for Metro Manila, April to May"
aircon,silip-domain skill (evening aircon scaled by heat index)
ev,charger kW and 8 PM start: silip-domain skill. Context: EVs were 24.7% of Jan to Aug 2026 vehicle sales (SPEC#sources); home share is [VERIFY]
solar,[VERIFY] net-metering uptake in the Meralco franchise
jumpers,[VERIFY] SPEC open question: which transformers have check meters
clamps,noise: SPEC#hardware (SCT-013-000 ±1%); the rest [VERIFY] from the pilot
transformers,"ratings, homes and base kW: silip-domain skill; loading mix and lead-up [VERIFY]"
outages,"trip rule and medians: silip-domain skill (restoration lognormal, median 2 to 4 h); weather multipliers [VERIFY]"


## Time grid, calendar and weather
One region, one weather series. Heat index peaks mid-afternoon; hot spells last several days; rain and typhoon days are cooler.

In [4]:
rng = np.random.default_rng(SEED)
days = config["days"]
ts = pd.date_range(START, periods=days * STEPS_PER_DAY, freq="15min")
hour = ts.hour.to_numpy() + ts.minute.to_numpy() / 60
day_index = np.repeat(np.arange(days), STEPS_PER_DAY)
holidays = pd.to_datetime(SCENARIOS["calendar"]["holidays"]).date
is_holiday = np.isin(ts.date, holidays)
home_day = is_holiday | (ts.day_of_week.to_numpy() >= 5)  # weekend or holiday

w = SCENARIOS["weather"]
weather_day = np.array(["clear"] * days, dtype=object)
for _ in range(max(1, round(days / 30 * w["hot_spells_per_30_days"]))):
    length = int(rng.integers(w["hot_spell_days"][0], w["hot_spell_days"][1] + 1))
    first = int(rng.integers(0, max(1, days - length)))
    weather_day[first : first + length] = "hot_spell"
weather_day[(rng.random(days) < w["rain_day_share"]) & (weather_day == "clear")] = "rain"
if days > 40:  # typhoon on a day that is not already a hot spell
    calm = np.flatnonzero(weather_day[40:] != "hot_spell") + 40
    weather_day[rng.choice(calm, w["typhoon_days_after_day_40"], replace=False)] = "typhoon"
daily_max = rng.normal(*w["daily_max_heat_index_c"], days)
daily_max += np.select(
    [weather_day == "hot_spell", weather_day == "rain", weather_day == "typhoon"],
    [w["hot_spell_extra_c"], -w["rain_cooling_c"], -w["typhoon_cooling_c"]],
    0.0,
)
diurnal = 0.5 + 0.5 * np.cos(2 * np.pi * (hour - 14) / 24)  # 1 at 2 PM, 0 at 2 AM
heat = daily_max[day_index] - w["day_night_swing_c"] * (1 - diurnal) + rng.normal(0, 0.5, len(ts))
afternoon = (hour >= 13) & (hour < 19)
rain = np.select(
    [weather_day[day_index] == "typhoon", (weather_day[day_index] == "rain") & afternoon],
    [rng.uniform(10, 30, len(ts)), rng.uniform(2, 10, len(ts))],
    0.0,
)
weather = pd.DataFrame(
    {"ts": ts, "heat_index_c": heat.round(2), "rain_mm_h": rain.round(1),
     "weather": weather_day[day_index], "is_holiday": is_holiday}
)
weather.groupby(weather["ts"].dt.date)["weather"].first().value_counts()

weather
clear        37
hot_spell    14
rain          4
typhoon       1
Name: count, dtype: int64

## Sites and homes on real OSM buildings
Transformers sit on real OSM sites, picked by seed. Homes sit on the nearest free buildings, shared as units when there are too few. The `large` scales need more transformers than the snapshot has; they reuse sites shifted about 1 km per copy (positions do not affect timing).

In [5]:
osm_sites = pd.read_csv(ROOT / "data/osm/transformers.csv", keep_default_na=False)
osm_buildings = pd.read_csv(ROOT / "data/osm/buildings.csv")
count = config["transformers"]
copies = -(-count // len(osm_sites))
if copies == 1:
    sites = osm_sites.iloc[np.random.default_rng(SEED).choice(len(osm_sites), count, replace=False)]
else:
    tiled = [osm_sites.assign(lat=osm_sites.lat + 0.01 * c, lon=osm_sites.lon + 0.01 * c, copy=c) for c in range(copies)]
    sites = pd.concat(tiled).iloc[:count]
sites = sites.reset_index(drop=True)
METERS_PER_DEG_LAT, SEARCH_RADIUS_M, SHARED_UNIT_JITTER_M = 111_320, 150, 4


def place_homes(rng, homes, lat, lon, near, taken):
    """Nearest free OSM buildings first; units share a building when there are too few."""
    free = near[~near["osm_way_id"].isin(taken)]
    pool = free if len(free) > 0 else near
    if len(pool) == 0:  # no building nearby: scatter within the search radius
        radius = SEARCH_RADIUS_M * np.sqrt(rng.random(homes))
        angle = rng.uniform(0, 2 * np.pi, homes)
        dy, dx = radius * np.sin(angle), radius * np.cos(angle)
        way = pd.array([pd.NA] * homes, dtype="Int64")
        base_lat, base_lon = np.full(homes, lat), np.full(homes, lon)
    else:
        picks = pool.iloc[np.arange(homes) % len(pool)]
        taken.update(picks["osm_way_id"].astype(int))
        shared = np.arange(homes) >= len(pool)
        dy, dx = rng.normal(0, SHARED_UNIT_JITTER_M, (2, homes)) * shared
        way = pd.array(picks["osm_way_id"].to_numpy(), dtype="Int64")
        base_lat, base_lon = picks["lat"].to_numpy(), picks["lon"].to_numpy()
    k_lon = METERS_PER_DEG_LAT * np.cos(np.radians(lat))
    return base_lat + dy / METERS_PER_DEG_LAT, base_lon + dx / k_lon, way


def sin_hash(raw_sin: str) -> str:
    """Salted SHA-256, as in silip_ml.core.privacy. The raw SIN exists only inside this call."""
    return hashlib.sha256((SALT + raw_sin).encode()).hexdigest()

## One household's day
Base load with a morning and an evening bump, more daytime use on weekends and holidays, evening and night aircon scaled by heat index (silip-domain recipe), sensor-level noise. EV homes start charging at 8 PM from the day they adopt. Solar homes subtract rooftop output at midday and can go negative.

In [6]:
morning = np.exp(-(((hour - 7) / 1.2) ** 2))
evening = np.exp(-(((hour - 20.5) / 2.0) ** 2))
def window(on: float, off: float, ramp_h: float = 0.5) -> np.ndarray:
    """Smooth 0-to-1 window from hour `on` to hour `off`, wrapping past midnight."""
    length = (off - on) % 24
    from_centre = (hour - (on + length / 2) + 12) % 24 - 12
    return 1 / (1 + np.exp(-(length / 2 - np.abs(from_centre)) / ramp_h))


night_ac = window(18, 2)  # evening and night aircon
daytime = window(9, 17)
heat_scale = np.clip((heat - 30) / 10, 0, 1.5)
sun = np.clip(np.sin(np.pi * (hour - 6) / 12), 0, None)
cloud_by_day = np.where(weather_day == "typhoon", 0.1, np.where(weather_day == "rain", 0.3, 0.85))


def homes_kw(rng, homes):
    """kW per home and interval, shape (homes, steps). Returns the extras for the household table."""
    steps = len(ts)
    base = rng.uniform(*SCENARIOS["transformers"]["base_kw"], (homes, 1))
    shape = 0.8 + 0.4 * morning + 0.6 * evening
    extra = SCENARIOS["calendar"]["home_daytime_extra_kw"] * home_day * daytime
    aircon_kw = rng.uniform(*SCENARIOS["aircon"]["kw_per_home"], (homes, 1))
    aircon = aircon_kw * heat_scale * (night_ac + 0.5 * daytime * home_day)
    kw = base * shape + extra + aircon + rng.normal(0, 0.05, (homes, steps))
    return np.clip(kw, 0.05, None)


def add_ev(rng, kw, share_by_end):
    """Homes adopt an EV at a random day and charge from 8 PM most nights. Returns adoption day per home (-1 none)."""
    homes, e = kw.shape[0], SCENARIOS["ev"]
    adopt_day = np.full(homes, -1)
    adopters = np.flatnonzero(rng.random(homes) < share_by_end)
    for h in adopters:
        adopt_day[h] = int(rng.integers(0, days))
        charger, hours = rng.uniform(*e["charger_kw"]), rng.uniform(*e["charge_hours"])
        since_eight = (hour - e["charge_from_hour"]) % 24
        charging = (since_eight < hours) & (day_index >= adopt_day[h])
        charging &= rng.random(days)[day_index] < 0.8  # not every night
        kw[h] += charger * charging
    return adopt_day


def add_solar(rng, kw):
    """Some homes subtract rooftop output; net kW can go below zero at midday."""
    homes = kw.shape[0]
    kwp = np.where(rng.random(homes) < SCENARIOS["solar"]["home_share"], rng.uniform(*SCENARIOS["solar"]["kwp"], homes), 0.0)
    kw -= kwp[:, None] * sun * cloud_by_day[day_index] * rng.uniform(0.9, 1.0, len(ts))
    return kwp

## Transformer size and loading class
Real transformers are sized for the homes on them, so most run below their rating and a few run hot. Each transformer draws a loading class, then the generator picks the rating and home count (within the silip-domain ranges) whose 99th-percentile load lands closest to that class. A few transformers get a lead-up: load grows over the last 1 to 2 weeks (new homes, new aircon) until they trip.

In [7]:
t_cfg = SCENARIOS["transformers"]
classes = list(t_cfg["loading_mix"])
class_share = np.array([t_cfg["loading_mix"][c][0] for c in classes])


def size_transformer(rng, kw_all):
    """Pick (rating, homes) so the first two weeks' 99th-percentile load hits the drawn class."""
    loading_class = classes[rng.choice(len(classes), p=class_share / class_share.sum())]
    _, low, high = t_cfg["loading_mix"][loading_class]
    target = rng.uniform(low, high)
    window = min(len(ts), 14 * STEPS_PER_DAY)
    totals = np.cumsum(kw_all[:, :window], axis=0)  # row n-1 = sum of the first n homes
    n_range = np.arange(t_cfg["homes"][0], t_cfg["homes"][1] + 1)
    peak = np.percentile(totals[n_range - 1], 99, axis=1)
    best = None
    for rated in rng.permutation(t_cfg["ratings_kva"]):
        error = np.abs(peak / (rated * 0.9) - target)
        i = int(error.argmin())
        if best is None or error[i] < best[0]:
            best = (error[i], float(rated), int(n_range[i]))
    return loading_class, best[1], best[2]

## Outages
Overload trips follow the silip-domain rule on the true load. Weather outages are more likely on rain and typhoon days. While power is out, true load is zero and clamps send nothing.

In [8]:
o_cfg = SCENARIOS["outages"]
TRIP_PCT, TRIP_STEPS, TRIP_DAYS = 1.10, 12, 3
day_rate = o_cfg["other_per_day"] * np.select(
    [weather_day == "rain", weather_day == "typhoon"], [o_cfg["rain_multiplier"], o_cfg["typhoon_multiplier"]], 1
)


def trip_start(pct):
    """Index where the 3-hour run above 110% completes on the third such day, or None."""
    over = pd.Series(pct > TRIP_PCT).rolling(TRIP_STEPS).min().fillna(0).astype(bool).to_numpy()
    flagged_days = np.unique(day_index[over])
    if len(flagged_days) < TRIP_DAYS:
        return None
    return int(np.flatnonzero(over & (day_index == flagged_days[TRIP_DAYS - 1]))[0])


def outages_for(rng, true_pct):
    """List of (start index, cause, minutes)."""
    found = []
    if (i := trip_start(true_pct)) is not None:
        found.append((i, "overload"))
    for d in range(days):
        for _ in range(rng.poisson(day_rate[d] / 2)):  # half weather, half equipment
            found.append((d * STEPS_PER_DAY + int(rng.integers(STEPS_PER_DAY)), "weather"))
        for _ in range(rng.poisson(o_cfg["other_per_day"] / 2)):
            found.append((d * STEPS_PER_DAY + int(rng.integers(STEPS_PER_DAY)), "equipment"))
    rows = []
    for i, cause in sorted(found):
        median = o_cfg["median_minutes"][cause] * (2 if weather_day[day_index[i]] == "typhoon" else 1)
        rows.append((i, cause, round(float(rng.lognormal(np.log(median), o_cfg["sigma"])))))
    return rows

## Clamp reality
Opt-in coverage differs per transformer. Some clamps are installed mid-run, some go offline for good, home Wi-Fi drops for hours, single intervals go missing, a few readings arrive late or twice, and each reading carries ±1% sensor noise. SILIP sees only what the clamps send.

In [9]:
c_cfg = SCENARIOS["clamps"]


def clamp_mask(rng, clamped, power_off):
    """True where a clamped home's reading for that interval reached SILIP. Shape (clamped, steps)."""
    steps = len(ts)
    first = np.where(rng.random(clamped) < c_cfg["installed_during_run_share"], rng.integers(0, steps // 2, clamped), 0)
    last = np.where(rng.random(clamped) < c_cfg["offline_share"], rng.integers(steps // 2, steps, clamped), steps)
    step = np.arange(steps)
    mask = (step >= first[:, None]) & (step < last[:, None])
    for c in range(clamped):
        for _ in range(rng.poisson(c_cfg["wifi_drops_per_day"] * days)):
            start = int(rng.integers(steps))
            mask[c, start : start + int(rng.uniform(*c_cfg["wifi_drop_hours"]) * 4)] = False
    mask &= rng.random((clamped, steps)) >= c_cfg["missing_interval_share"]
    return mask & ~power_off, first, last

## Generate every transformer
One random stream per transformer (`[seed, t]`), so the dev set is a prefix of nothing and every transformer is reproducible on its own.

In [10]:
started = time.perf_counter()
tx_rows, sin_frames, load_frames, outage_rows, signal_rows, clamp_frames, detail_frames = [], [], [], [], [], [], []
detail_ids, ev_sites = {}, np.random.default_rng([SEED, 999]).random(count) < SCENARIOS["ev"]["site_share"]
taken: set[int] = set()
step_all = np.arange(len(ts))
for t in range(count):
    rng_t = np.random.default_rng([SEED, t])
    tid, site = f"T-{100 + t}", sites.iloc[t]
    lat, lon = float(site["lat"]), float(site["lon"])
    kw_all = homes_kw(rng_t, t_cfg["homes"][1])
    loading_class, rated, homes = size_transformer(rng_t, kw_all)
    kw = kw_all[:homes].copy()
    ev_share = SCENARIOS["ev"]["home_share_by_end"]["ev_site" if ev_sites[t] else "other"]
    adopt_day = add_ev(rng_t, kw, ev_share)
    kwp = add_solar(rng_t, kw)
    lead_up = rng_t.random() < t_cfg["lead_up_share"] and loading_class != "light"
    if lead_up:
        ramp_days = rng_t.integers(*t_cfg["lead_up_days"])
        ramp = np.clip((step_all - (len(ts) - ramp_days * STEPS_PER_DAY)) / (ramp_days * STEPS_PER_DAY), 0, 1)
        kw *= 1 + rng_t.uniform(*t_cfg["lead_up_growth"]) * ramp
    has_jumper = rng_t.random() < SCENARIOS["jumpers"]["transformer_share"]
    jumper_kw = rng_t.uniform(*SCENARIOS["jumpers"]["kw"]) * (0.3 + evening) if has_jumper else np.zeros(len(ts))
    true_kw = kw.sum(axis=0) + jumper_kw
    outages = outages_for(rng_t, true_kw / (rated * 0.9))
    power_off = np.zeros(len(ts), dtype=bool)
    for i, _, minutes in outages:
        power_off[i : i + max(1, minutes // 15)] = True
    true_kw = np.where(power_off, 0.0, true_kw)
    kw[:, power_off] = 0.0

    # What the clamps report.
    coverage = rng_t.uniform(*c_cfg["coverage"])
    has_clamp = rng_t.random(homes) < coverage
    clamped = np.flatnonzero(has_clamp)
    mask, first, last = clamp_mask(rng_t, len(clamped), power_off)
    if t == 0 and len(clamped):  # the household view's demo home: a clamp that never drops out
        mask[0], first[0], last[0] = ~power_off, 0, len(ts)
    noisy = kw[clamped] * (1 + rng_t.normal(0, c_cfg["noise_share"], mask.shape))
    reporting = mask.sum(axis=0)
    measured = np.where(mask, noisy, 0.0).sum(axis=0)
    keep = reporting > 0  # no clamp reported: the interval is missing
    transformer_kw = measured[keep] * homes / reporting[keep]  # scale up for unmetered homes
    has_check_meter = rng_t.random() < SCENARIOS["jumpers"]["check_meter_share"]
    check = true_kw * (1 + rng_t.normal(0, 0.005, len(ts))) if has_check_meter else np.full(len(ts), np.nan)
    load_frames.append(pd.DataFrame({
        "transformer_id": tid, "ts": ts[keep], "transformer_kw": transformer_kw.round(3),
        "load_pct": (transformer_kw / (rated * 0.9)).round(4), "heat_index_c": heat[keep].round(2),
        "measured_kw": measured[keep].round(3), "homes_reporting": reporting[keep],
        "late_readings": rng_t.binomial(reporting[keep], c_cfg["late_share"]),
        "duplicate_readings": rng_t.binomial(reporting[keep], c_cfg["duplicate_share"]),
        "true_kw": true_kw[keep].round(3), "check_meter_kw": check[keep].round(3),
    }))

    hashes = [sin_hash(f"{tid}-{h}") for h in range(homes)]
    place_lat, place_lon, way = place_homes(
        np.random.default_rng([SEED, t, 1]), homes, lat, lon,
        osm_buildings[osm_buildings["osm_node_id"] == site["osm_node_id"]] if site.get("copy", 0) == 0 else osm_buildings.iloc[:0],
        taken,
    )
    sin_frames.append(pd.DataFrame({
        "sin_hash": hashes, "transformer_id": tid, "has_clamp": has_clamp, "lat": place_lat, "lon": place_lon,
        "osm_way_id": way, "has_ev": adopt_day >= 0,
        "ev_from": [ts[d * STEPS_PER_DAY].date() if d >= 0 else None for d in adopt_day],
        "solar_kwp": kwp.round(1),
    }))
    seen = [ts[np.flatnonzero(row)[-1]] if row.any() else ts[f] for row, f in zip(mask, first)]
    clamp_frames.append(pd.DataFrame({
        "sin_hash": [hashes[h] for h in clamped], "ct_model": "SCT-013-000",
        "installed_at": [(ts[f] - pd.Timedelta(days=float(rng_t.uniform(1, 120)))).floor("min") if f == 0 else ts[f] for f in first],
        "last_seen_at": [(s + pd.Timedelta(seconds=float(rng_t.uniform(0, 60)))).floor("s") for s in seen],
    }))
    tx_rows.append({
        "transformer_id": tid, "rated_kva": rated, "homes": homes, "coverage_pct": has_clamp.mean(),
        "lat": lat, "lon": lon, "osm_node_id": int(site["osm_node_id"]) if site.get("copy", 0) == 0 else None,
        "loading_class": loading_class, "ev_site": bool(ev_sites[t]), "ev_homes": int((adopt_day >= 0).sum()),
        "solar_homes": int((kwp > 0).sum()), "has_jumper": has_jumper, "has_check_meter": has_check_meter,
        "lead_up": lead_up,
    })
    for i, cause, minutes in outages:
        outage_id = f"O-{len(outage_rows) + 1:04d}"
        start = ts[i]
        outage_rows.append({"outage_id": outage_id, "transformer_id": tid, "start_ts": start, "cause": cause,
                            "duration_min": minutes, "lat": lat, "lon": lon})
        online = int(mask[:, max(i - 1, 0)].sum())  # clamps alive just before power went out
        for _ in range(online):  # last gasp within a minute
            signal_rows.append({"outage_id": outage_id, "kind": "last_gasp", "ts": start + pd.Timedelta(seconds=float(rng_t.uniform(0, 60))),
                                "lat": lat + rng_t.normal(0, 0.0003), "lon": lon + rng_t.normal(0, 0.0003)})
        for _ in range(1 + rng_t.poisson(3)):  # customer reports trickle in
            signal_rows.append({"outage_id": outage_id, "kind": "report", "ts": start + pd.Timedelta(seconds=float(rng_t.exponential(600))),
                                "lat": lat + rng_t.normal(0, 0.0015), "lon": lon + rng_t.normal(0, 0.0015)})

    # Per-home detail for a few transformers (household view, EDA): T-100, then the first solar and EV sites.
    reason = (
        "first" if t == 0
        else "solar" if (kwp > 0).any() and "solar" not in detail_ids.values()
        else "ev" if ev_sites[t] and (adopt_day >= 0).any() and "ev" not in detail_ids.values()
        else None
    )
    if reason and len(detail_ids) < config["detail"]:
        detail_ids[tid] = reason
        steps = slice(None) if config["detail_days"] is None else slice(len(ts) - config["detail_days"] * STEPS_PER_DAY, None)
        h_idx, s_idx = np.nonzero(mask[:, steps])
        offset = 0 if steps.start is None else steps.start
        detail_frames.append(pd.DataFrame({
            "sin_hash": np.array(hashes)[clamped][h_idx], "transformer_id": tid, "ts": ts[s_idx + offset],
            "power_kw": noisy[h_idx, s_idx + offset].round(3),
        }))
    if t % 500 == 499:
        print(f"{t + 1} transformers, {time.perf_counter() - started:.0f} s")
print(f"generated {count} transformers in {time.perf_counter() - started:.1f} s; per-home detail: {detail_ids}")

generated 50 transformers in 1.7 s; per-home detail: {'T-100': 'first', 'T-102': 'ev', 'T-104': 'solar'}


## Raw readings with late and duplicate arrivals
For the first detail transformer, the last two days as they reached the broker: each reading has a `received_at`, a few arrive after their 15-minute window closed, and a few arrive twice. `household_load_15m.csv` is the cleaned version (one row per home and interval).

In [11]:
households = pd.concat(detail_frames, ignore_index=True)
raw = households[(households["transformer_id"] == "T-100") & (households["ts"] >= ts[-1] - pd.Timedelta(days=2))].copy()
raw_rng = np.random.default_rng([SEED, 2])
delay_s = raw_rng.exponential(20, len(raw))
late = raw_rng.random(len(raw)) < c_cfg["late_share"]
delay_s[late] += raw_rng.uniform(15 * 60, 6 * 3600, late.sum())
raw["received_at"] = (raw["ts"] + pd.Timedelta(minutes=15) + pd.to_timedelta(delay_s, unit="s")).dt.floor("s")
dupes = raw.sample(frac=c_cfg["duplicate_share"], random_state=SEED)
readings_raw = pd.concat([raw, dupes.assign(received_at=dupes["received_at"] + pd.Timedelta(seconds=30))])
readings_raw = readings_raw.sort_values(["received_at", "sin_hash"], ignore_index=True)
print(f"raw readings: {len(readings_raw)} rows, {late.mean():.1%} late, {len(dupes)} duplicates")

raw readings: 4424 rows, 1.7% late, 22 duplicates


## Write the tables
Same file names and columns as before; new columns are added at the end, and three tables are new (`weather_15m`, `readings_raw`, plus scenario columns). Clamp IDs follow the firmware's MQTT client ID.

In [ ]:
clamps = pd.concat(clamp_frames, ignore_index=True)
clamps.insert(0, "clamp_id", [f"silip-node-{i:04d}" for i in range(1, len(clamps) + 1)])
transformers = pd.DataFrame(tx_rows)
transformers["osm_node_id"] = transformers["osm_node_id"].astype("Int64")
transformer_load = pd.concat(load_frames, ignore_index=True)
sin_map = pd.concat(sin_frames, ignore_index=True)

# Temporary dev-map anchor for the registered T-Meralco transformer. Its location and rating
# match the registry; its load profile is synthetic and remains labeled is_simulated=True.
if SCALE == "dev":
    demo_id = "T-Meralco"
    if demo_id in set(transformers["transformer_id"]):
        raise ValueError(f"{demo_id} already exists in the generated sample")
    donor = transformers.loc[transformers["rated_kva"] == 100].iloc[0]
    demo_transformer = donor.copy()
    demo_transformer["transformer_id"] = demo_id
    demo_transformer["lat"] = 14.591548975193037
    demo_transformer["lon"] = 121.06731718182328
    demo_transformer["osm_node_id"] = pd.NA
    transformers = pd.concat([transformers, pd.DataFrame([demo_transformer])], ignore_index=True)
    transformers["osm_node_id"] = transformers["osm_node_id"].astype("Int64")

    demo_load = transformer_load.loc[
        transformer_load["transformer_id"] == donor["transformer_id"]
    ].copy()
    demo_load["transformer_id"] = demo_id
    perturbation = np.random.default_rng([SEED, 20_260_112]).normal(1.0, 0.015, len(demo_load))
    for column in ("transformer_kw", "measured_kw", "true_kw", "check_meter_kw"):
        demo_load[column] = (demo_load[column] * perturbation).round(3)
    demo_load["load_pct"] = (demo_load["transformer_kw"] / (100 * 0.9)).round(4)
    transformer_load = pd.concat([transformer_load, demo_load], ignore_index=True)

    # Simulated map point for the physical node's ID. This is a fixture row only; the real
    # registration and readings stay in Neon and keep their real-data flags.
    demo_clamp_id = "silip-esp32-test"
    demo_sin_hash = sin_hash(f"{demo_id}-{demo_clamp_id}")
    if demo_clamp_id in set(clamps["clamp_id"]) or demo_sin_hash in set(sin_map["sin_hash"]):
        raise ValueError(f"{demo_clamp_id} already exists in the generated sample")
    sin_map = pd.concat(
        [
            sin_map,
            pd.DataFrame([{
                "sin_hash": demo_sin_hash, "transformer_id": demo_id, "has_clamp": True,
                "lat": demo_transformer["lat"], "lon": demo_transformer["lon"],
                "osm_way_id": pd.NA, "has_ev": False, "ev_from": None, "solar_kwp": 0.0,
            }]),
        ],
        ignore_index=True,
    )
    clamps = pd.concat(
        [
            clamps,
            pd.DataFrame([{
                "clamp_id": demo_clamp_id, "sin_hash": demo_sin_hash, "ct_model": "SCT-013-000",
                "installed_at": (ts[0] - pd.Timedelta(days=10)).floor("min"),
                "last_seen_at": ts[-1] + pd.Timedelta(seconds=14),
            }]),
        ],
        ignore_index=True,
    )

tables = {
    "transformers.csv": transformers,
    "sin_map.csv": sin_map,
    "clamps.csv": clamps,
    "transformer_load_15m.csv": transformer_load,
    "household_load_15m.csv": households,
    "readings_raw.csv": readings_raw,
    "weather_15m.csv": weather,
    "outages.csv": pd.DataFrame(outage_rows),
    "outage_signals.csv": pd.DataFrame(signal_rows, columns=["outage_id", "kind", "ts", "lat", "lon"]),
}
OUT.mkdir(parents=True, exist_ok=True)
for name, frame in tables.items():
    frame.assign(is_simulated=True).to_csv(OUT / name, index=False)
summary = pd.DataFrame(
    {"rows": {n: len(f) for n, f in tables.items()},
     "MB": {n: round((OUT / n).stat().st_size / 2**20, 2) for n in tables}}
)
summary

## Quick checks
Every row labeled simulated, every SIN a 64-hex hash, the scenario mix present.

In [13]:
sin_map = tables["sin_map.csv"]
assert sin_map["sin_hash"].str.fullmatch(r"[0-9a-f]{64}").all()
assert clamps["sin_hash"].is_unique and set(clamps["sin_hash"]) == set(sin_map.loc[sin_map["has_clamp"], "sin_hash"])
assert all(pd.read_csv(OUT / n, usecols=["is_simulated"])["is_simulated"].all() for n in tables)
transformers[["loading_class", "ev_site", "has_jumper", "has_check_meter", "lead_up"]].apply(
    lambda c: c.value_counts().to_dict()
).to_frame("count")

,count
loading_class,"{'normal': 22, 'light': 15, 'over': 7, 'tight'..."
ev_site,"{False: 38, True: 12}"
has_jumper,"{False: 42, True: 8}"
has_check_meter,"{False: 36, True: 14}"
lead_up,"{False: 43, True: 7}"


**Outcome.** The tables above are on disk. `01_data_overview` describes them; the scenario knobs and their sources are listed in `data/README.md`.